In [1]:
from pathlib import Path
import sys
from datetime import datetime

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "ruleset_comparison"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_ID = f"exp_ruleset_comparison_{RUN_TS}"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("EXPERIMENT_ID:", EXPERIMENT_ID)

PROJECT_ROOT: /home/harielpadillasanchez/Documentos/TT/TT2
DATA_DIR: /home/harielpadillasanchez/Documentos/TT/TT2/data
OUTPUT_DIR: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison
EXPERIMENT_ID: exp_ruleset_comparison_20260404_031154


In [2]:
import json
import pandas as pd

from configs.models import MODELS
from configs.rules import RULESETS
from src.experiment.runner import ExperimentRunner
from src.evaluation.metrics import evaluate_dataframe, summarize_metrics

/home/harielpadillasanchez/Documentos/TT/TT2/.venv-bloom/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
SAMPLE_PATH = DATA_DIR / "Muestra_csv.csv"

df_sample = pd.read_csv(SAMPLE_PATH)

print("Shape:", df_sample.shape)
print("Columnas:", list(df_sample.columns))
display(df_sample.head(3))

Shape: (20, 17)
Columnas: ['id', 'idFinal', 'grupo', 'motivo', 'Segmento', 'Propuesta', 'idcod', 'atr0', 'atr1', 'atr2', 'atr3', 'atr4', 'atr5', 'atr6', 'atr7', 'atr8', 'lex']


,id,idFinal,grupo,motivo,Segmento,Propuesta,idcod,atr0,atr1,atr2,atr3,atr4,atr5,atr6,atr7,atr8,lex
0,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,Vivian,10,1,4,5.0,NaN,NaN,NaN,NaN,NaN,1
1,2976,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,Vivian,14,5,1,6.0,NaN,NaN,NaN,NaN,NaN,1
2,3430,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,Uno de los problemas más importantes en los ne...,Asignar precios a los productos es un problema...,Vivian,5,6,19,1.0,NaN,NaN,NaN,NaN,NaN,1


In [4]:
META_COLS = ["idFinal", "grupo", "motivo", "lex"]

required_cols = ["id", "Segmento", "Propuesta"]
missing = [c for c in required_cols if c not in df_sample.columns]

if missing:
    raise ValueError(f"Faltan columnas requeridas en la muestra: {missing}")

df_refine = df_sample.copy()

df_refine = df_refine.rename(
    columns={
        "id": "sample_id",
        "Segmento": "source_text",
        "Propuesta": "reference_text",
    }
)

keep_cols = ["sample_id", "source_text", "reference_text"] + [c for c in META_COLS if c in df_refine.columns]
df_refine = df_refine[keep_cols].copy()

print("Shape refinamiento:", df_refine.shape)
display(df_refine.head(5))

Shape refinamiento: (20, 7)


,sample_id,source_text,reference_text,idFinal,grupo,motivo,lex
0,2088,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",1
1,2976,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,1
2,3430,Uno de los problemas más importantes en los ne...,Asignar precios a los productos es un problema...,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,1
3,3679,"El resultado será 190, o sea, el IVA del kilo ...","El resultado será ciento noventa. Es decir, el...",829_LibroUAC_Sincopyright.pdf,A_cortos,Corto con número e IVA.,1
4,3145,"Por cada $100 facturados, la compañía gasta $2...",La empresa gasta dos dólares con veinte centav...,1045_LibroUide_Sincopyright.pdf,A_cortos,Corto con proporción financiera.,1


In [5]:
FEW_SHOT_EXAMPLES = [
    {
        "source": """A continuación una tabla donde se demuestra como crecen los ahorros al pasar los años: Edad Inversión Intereses Saldo Inversión Intereses Saldo Laura Ganados Alberto Ganados 22 $800 $80 $880 23 $800 $168 $1.232 Valor al Jubilarse $311.092 Valor al Jubilarse $263.232 Menos las contribuciones iniciales $6.400 Menos las contribuciones iniciales $28.800 Ganancia Neta $304.692 Ganancia Neta $234.""",
        "target": """A continuación, presentamos una tabla donde se demuestra como crecen los ahorros al pasar los años. Las categorías en que se divide la tabla son edad, inversión, intereses ganados y saldo. La tabla contrasta las ganancias en intereses por año de Laura y Alberto con una proyección de edad de jubilación de sesenta y cinco años y según la cantidad de años que mantuvieron el ahorro."""
    },
    {
        "source": """El varias veces mencionado Yager, nos dice acertadamente sobre estos aspectos lo siguiente: La mayoría de ustedes están hambrientos de tener libertad financiera, están hastiados de tener batallas financieras porque son como un carrusel.""",
        "target": """Yager habla acertadamente sobre estos aspectos y dice que la mayoría ambiciona tener libertad financiera y está cansada de tener batallas financieras inútiles."""
    },
    {
        "source": """De acuerdo con esta medición, la proporción de pobres en la población mundial quienes viven con menos de un dólar por día descendió levemente entre 1987 y 1993, pues pasó del 30% al 29%.""",
        "target": """Según esta medición, la proporción de personas pobres en la población mundial descendió un poco entre mil novecientos ochenta y siete y mil novecientos noventa y tres. Es decir, el porcentaje de personas que vivían con menos de un dólar por día pasó del treinta al veintinueve por ciento."""
    },
    {
        "source": """- Rentabilidad
INTERÉS DE 2 PESOS SOBRE UNA INVERSIÓN DE 100 PESOS = RENTABILIDAD DE 2 % ((2 ÷ 100) × 100 = 2%) POR CADA 100 PESOS SE GANAN 2 PESOS
 Endeudamiento
 DEUDA DE 30 PESOS POR SOBRE EL CAPITAL DE 20 PESOS = ENDEUDAMIENTO DE 1,5 VECES EL CAPITAL (30 ÷ 20 = 1,5) POR CADA PESO DE CAPITAL EXISTE 1,5 PESOS DE DEUDA
 Liquidez
80 PESOS DE DINERO DISPONIBLE POR SOBRE DEUDAS DE 40 PESOS = LIQUIDEZ DE DOS VECES EL VALOR DE LA DEUDA (80 ÷ 40 = 2) POR CADA PESO DE DEUDA SE DISPONE DE 2 PESOS PARA PAGO""",
        "target": """EL INTERÉS DE DOS PESOS SOBRE UNA INVERSIÓN DE CIEN PESOS DA UNA RENTABILIDAD DE DOS POR CIENTO. ES DECIR, POR CADA CIEN PESOS GANAMOS DOS PESOS. LA DEUDA DE TREINTA PESOS SOBRE EL CAPITAL DE VEINTE PESOS RESULTA EN UNA DEUDA DE UNO PUNTO CINCO VECES EL CAPITAL. ES DECIR, POR CADA PESO DE CAPITAL HAY UNO PUNTO CINCO PESOS DE DEUDA. EN OCHENTA PESOS DE DINERO DISPONIBLE SOBRE DEUDAS DE CUARENTA PESOS DA UNA LIQUIDEZ DEL DOBLE DE LA DEUDA. POR CADA PESO ADEUDADO HAY DOS PESOS PARA PAGAR."""
    },
]

FEW_SHOT_EXAMPLE_IDS = [78, 1805, 3635, 5262]

print("Número de ejemplos few-shot:", len(FEW_SHOT_EXAMPLES))
print("IDs few-shot:", FEW_SHOT_EXAMPLE_IDS)

Número de ejemplos few-shot: 4
IDs few-shot: [78, 1805, 3635, 5262]


In [6]:

PROMPT_TYPE = "few-shot"

FINALIST_CONFIGS = [
    {
        "model_key": "llama3",
        "temperature": 0.3,
        "top_p": 0.90,
        "repetition_penalty": 1.15,
        "do_sample": True,
        "no_repeat_ngram_size": 4,
        "config_label": "llama3_cfg_1",
        "max_new_tokens": 256,
    },
    {
        "model_key": "llama3",
        "temperature": 0.7,
        "top_p": 0.90,
        "repetition_penalty": 1.1,
        "max_new_tokens": 512,
        "do_sample": True,
        "no_repeat_ngram_size": 4,
        "config_label": "llama3_cfg_2",
    },
    {
        "model_key": "mistral",
        "temperature": 0.7,
        "top_p": 0.90,
        "repetition_penalty": 1.1,
        "max_new_tokens": 512,
        "do_sample": True,
        "no_repeat_ngram_size": 4,
        "config_label": "mistral_cfg_1",
    },
    {
        "model_key": "mistral",
        "temperature": 0.3,
        "top_p": 0.90,
        "repetition_penalty": 1.15,
        "max_new_tokens": 400,
        "do_sample": True,
        "no_repeat_ngram_size": 4,
        "config_label": "mistral_cfg_2",
    },
]

ACTIVE_RULESETS = ["R0", "R1", "R2", "R3", "R4"]

print("Prompt type fijo:", PROMPT_TYPE)
print("N configuraciones finalistas:", len(FINALIST_CONFIGS))
print("Rulesets:", ACTIVE_RULESETS)
display(pd.DataFrame(FINALIST_CONFIGS))

Prompt type fijo: few-shot
N configuraciones finalistas: 4
Rulesets: ['R0', 'R1', 'R2', 'R3', 'R4']


,model_key,temperature,top_p,repetition_penalty,do_sample,no_repeat_ngram_size,config_label,max_new_tokens
0,llama3,0.3,0.9,1.15,True,4,llama3_cfg_1,256
1,llama3,0.7,0.9,1.10,True,4,llama3_cfg_2,512
2,mistral,0.7,0.9,1.10,True,4,mistral_cfg_1,512
3,mistral,0.3,0.9,1.15,True,4,mistral_cfg_2,400


In [7]:
for cfg in FINALIST_CONFIGS:
    if cfg["model_key"] not in MODELS:
        raise ValueError(f"Modelo no definido en MODELS: {cfg['model_key']}")

for ruleset in ACTIVE_RULESETS:
    if ruleset not in RULESETS:
        raise ValueError(f"Ruleset no definido en RULESETS: {ruleset}")

if PROMPT_TYPE not in ["zero-shot", "few-shot"]:
    raise ValueError(f"Técnica no soportada: {PROMPT_TYPE}")

print("Configuración validada correctamente.")

Configuración validada correctamente.


In [8]:
runner = ExperimentRunner(
    experiment_id=EXPERIMENT_ID,
    log_dir=str(PROJECT_ROOT / "outputs" / "logs")
)

print("Runner inicializado:", runner.experiment_id)

Runner inicializado: exp_ruleset_comparison_20260404_031154


In [9]:
from src.evaluation.metrics import compute_bertscore_batch

refs_test = [
    "La comunidad humana más antigua se llamó tribu.",
    "Asignar precios a los productos es un problema importante.",
]

preds_test = [
    "La comunidad humana más antigua se llama horda primitiva.",
    "Poner precios a los productos es un problema importante.",
]

try:
    bert_vals = compute_bertscore_batch(
        references=refs_test,
        predictions=preds_test,
        lang="es",
        model_type=None,
    )
    print("BERTScore values:", bert_vals)
    print("Tipos:", [type(x) for x in bert_vals])
except Exception as e:
    print("ERROR DIRECTO EN LA CELDA:", repr(e))

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

BERTScore values: [0.9127476215362549, 0.9741193056106567]
Tipos: [<class 'float'>, <class 'float'>]


In [10]:
import pandas as pd
from src.evaluation.metrics import evaluate_dataframe

df_test = pd.DataFrame({
    "source_text": [
        "La comunidad humana más antigua ha sido denominada horda primitiva.",
        "Uno de los problemas más importantes en los negocios es fijar precios.",
    ],
    "generated_text": [
        "La comunidad humana más antigua se llama horda primitiva.",
        "Uno de los problemas más importantes en los negocios es poner precios.",
    ],
    "reference_text": [
        "La comunidad humana más antigua se llamó tribu.",
        "Asignar precios a los productos es un problema importante.",
    ],
})

df_eval_test = evaluate_dataframe(
    df_test,
    source_col="source_text",
    pred_col="generated_text",
    ref_col="reference_text",
)

display(df_eval_test[[
    "sari",
    "rougeL_f",
    "bertscore_f1",
    "sbert_similarity"
]])

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

,sari,rougeL_f,bertscore_f1,sbert_similarity
0,67.760943,0.736842,0.912748,None
1,15.631868,0.181818,0.826148,None


In [11]:
test_row = df_refine.iloc[0]
test_cfg = FINALIST_CONFIGS[0]

test_record = runner.run_one(
    dataset_name="muestra_20_ruleset_comparison",
    model_key=test_cfg["model_key"],
    prompt_type=PROMPT_TYPE,
    ruleset=ACTIVE_RULESETS[0],
    source_text=test_row["source_text"],
    reference_text=test_row["reference_text"],
    sample_id=str(test_row["sample_id"]),
    fold_id=None,
    split_name="ruleset_comparison",
    few_shot_examples=FEW_SHOT_EXAMPLES if PROMPT_TYPE == "few-shot" else None,
    few_shot_example_ids=FEW_SHOT_EXAMPLE_IDS if PROMPT_TYPE == "few-shot" else None,
    generation_config={
        "temperature": test_cfg["temperature"],
        "top_p": test_cfg["top_p"],
        "repetition_penalty": test_cfg["repetition_penalty"],
        "max_new_tokens": test_cfg["max_new_tokens"],
    },
)

test_record.to_dict()

{'experiment_id': 'exp_ruleset_comparison_20260404_031154',
 'run_id': '028f1c07-f82c-4124-8537-8d1648725b73',
 'timestamp': '2026-04-04T03:12:00.858790',
 'dataset_name': 'muestra_20_ruleset_comparison',
 'fold_id': None,
 'split_name': 'ruleset_comparison',
 'model_key': 'llama3',
 'model_id': 'meta-llama/Meta-Llama-3-8B-Instruct',
 'backend': 'ollama',
 'prompt_type': 'few-shot',
 'ruleset': 'R0',
 'few_shot_example_ids': [78, 1805, 3635, 5262],
 'generation_config': {'temperature': 0.3,
  'top_p': 0.9,
  'repetition_penalty': 1.15,
  'max_new_tokens': 256},
 'sample_id': '2088',
 'source_text': 'La comunidad humana más antigua ha sido denominada horda primitiva.',
 'reference_text': 'La comunidad humana más antigua se llamó tribu.',
 'generated_text': 'La comunidad humana más antigua se llama horda primitiva.',
 'prompt_text': 'Reescribe en español cada texto con lenguaje más claro y sencillo.\nConserva el significado original y no inventes información.\n\nDevuelve solo la versión 

In [14]:
all_records = []

total_runs = len(FINALIST_CONFIGS) * len(ACTIVE_RULESETS) * len(df_refine)
run_counter = 0

for cfg in FINALIST_CONFIGS:
    for ruleset in ACTIVE_RULESETS:
        for _, row in df_refine.iterrows():
            run_counter += 1

            generation_config = {
                "temperature": cfg["temperature"],
                "top_p": cfg["top_p"],
                "repetition_penalty": cfg["repetition_penalty"],
                "max_new_tokens": cfg["max_new_tokens"],
            }

            print(
                f"[{run_counter}/{total_runs}] "
                f"model={cfg['model_key']} | cfg={cfg['config_label']} | "
                f"ruleset={ruleset} | sample_id={row['sample_id']}"
            )

            record = runner.run_one(
                dataset_name="muestra_20_ruleset_comparison",
                model_key=cfg["model_key"],
                prompt_type=PROMPT_TYPE,
                ruleset=ruleset,
                source_text=row["source_text"],
                reference_text=row["reference_text"],
                sample_id=str(row["sample_id"]),
                fold_id=None,
                split_name="ruleset_comparison",
                few_shot_examples=FEW_SHOT_EXAMPLES if PROMPT_TYPE == "few-shot" else None,
                few_shot_example_ids=FEW_SHOT_EXAMPLE_IDS if PROMPT_TYPE == "few-shot" else None,
                generation_config=generation_config,
            )

            record_dict = record.to_dict()
            record_dict["config_label"] = cfg["config_label"]

            for meta_col in ["idFinal", "grupo", "motivo", "lex"]:
                if meta_col in row.index:
                    record_dict[meta_col] = row[meta_col]

            all_records.append(record_dict)

print(f"Corridas completadas: {len(all_records)}")

[1/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=2088
[2/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=2976
[3/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3430
[4/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3679
[5/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3145
[6/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=507
[7/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=1756
[8/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3093
[9/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3192
[10/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3525
[11/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3627
[12/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3206
[13/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=329
[14/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3481
[15/400] model=ll

In [15]:
results_df = pd.DataFrame(all_records)

print("Shape results_df:", results_df.shape)
display(results_df.head(3))

if "status" in results_df.columns:
    display(results_df["status"].value_counts(dropna=False))

Shape results_df: (400, 27)


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,prompt_text,inference_seconds,status,error_message,metrics,config_label,idFinal,grupo,motivo,lex
0,exp_ruleset_comparison_20260404_031154,5fdf42e4-31e8-4817-ac03-1487603ba5be,2026-04-04T03:12:22.621812,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Reescribe en español cada texto con lenguaje m...,4.0027,success,None,{},llama3_cfg_1,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",1
1,exp_ruleset_comparison_20260404_031154,0825c2fb-3816-49a7-825b-9a51e93671b4,2026-04-04T03:12:26.788447,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Reescribe en español cada texto con lenguaje m...,6.1234,success,None,{},llama3_cfg_1,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,1
2,exp_ruleset_comparison_20260404_031154,60972094-8261-4502-8fc9-4fd8d8ec92b8,2026-04-04T03:12:33.086174,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Reescribe en español cada texto con lenguaje m...,5.6045,success,None,{},llama3_cfg_1,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,1


status
success    400
Name: count, dtype: int64

In [16]:
raw_results_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_raw_results.csv"
results_df.to_csv(raw_results_path, index=False, encoding="utf-8-sig")

print(raw_results_path)

/home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260404_031154_raw_results.csv


In [17]:
successful_df = results_df[results_df["status"] == "success"].copy()

print("Corridas exitosas:", len(successful_df))
print("Corridas con error:", len(results_df) - len(successful_df))
display(successful_df.head(3))

Corridas exitosas: 400
Corridas con error: 0


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,prompt_text,inference_seconds,status,error_message,metrics,config_label,idFinal,grupo,motivo,lex
0,exp_ruleset_comparison_20260404_031154,5fdf42e4-31e8-4817-ac03-1487603ba5be,2026-04-04T03:12:22.621812,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Reescribe en español cada texto con lenguaje m...,4.0027,success,None,{},llama3_cfg_1,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",1
1,exp_ruleset_comparison_20260404_031154,0825c2fb-3816-49a7-825b-9a51e93671b4,2026-04-04T03:12:26.788447,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Reescribe en español cada texto con lenguaje m...,6.1234,success,None,{},llama3_cfg_1,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,1
2,exp_ruleset_comparison_20260404_031154,60972094-8261-4502-8fc9-4fd8d8ec92b8,2026-04-04T03:12:33.086174,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Reescribe en español cada texto con lenguaje m...,5.6045,success,None,{},llama3_cfg_1,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,1


In [18]:
evaluated_df = evaluate_dataframe(
    successful_df,
    source_col="source_text",
    pred_col="generated_text",
    ref_col="reference_text",
)

print("Shape evaluated_df:", evaluated_df.shape)
display(evaluated_df.head(3))

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 6d86ad01-77d1-4f6c-b525-20c23701fbd1)')' thrown while requesting HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamm

Shape evaluated_df: (400, 52)


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,additions_proportion,deletions_proportion,inflesz_pred,inflesz_src,inflesz_delta,rouge1_f,rouge2_f,rougeL_f,bertscore_f1,sbert_similarity
0,exp_ruleset_comparison_20260404_031154,5fdf42e4-31e8-4817-ac03-1487603ba5be,2026-04-04T03:12:22.621812,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,0.222222,0.300000,52.468333,34.855000,17.613333,0.736842,0.705882,0.736842,0.912748,None
1,exp_ruleset_comparison_20260404_031154,0825c2fb-3816-49a7-825b-9a51e93671b4,2026-04-04T03:12:26.788447,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,0.750000,0.750000,85.703750,105.172500,-19.468750,0.062500,0.000000,0.062500,0.740974,None
2,exp_ruleset_comparison_20260404_031154,60972094-8261-4502-8fc9-4fd8d8ec92b8,2026-04-04T03:12:33.086174,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,0.470588,0.470588,57.905588,76.229118,-18.323529,0.333333,0.071429,0.200000,0.815382,None


In [19]:
param_cols = evaluated_df["generation_config"].apply(pd.Series)
param_cols = param_cols.rename(columns=lambda c: f"gen_{c}")

evaluated_df = pd.concat([evaluated_df, param_cols], axis=1)

display(
    evaluated_df[
        [
            "model_key",
            "config_label",
            "ruleset",
            "gen_temperature",
            "gen_top_p",
            "gen_repetition_penalty",
            "gen_max_new_tokens",
        ]
    ].head(5)
)

,model_key,config_label,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens
0,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,256.0
1,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,256.0
2,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,256.0
3,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,256.0
4,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,256.0


In [20]:
evaluated_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_evaluated.csv"
evaluated_df.to_csv(evaluated_path, index=False, encoding="utf-8-sig")

print(evaluated_path)

/home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260404_031154_evaluated.csv


In [21]:
summary_by_ruleset = summarize_metrics(
    evaluated_df,
    group_cols=[
        "model_key",
        "config_label",
        "ruleset",
        "gen_temperature",
        "gen_top_p",
        "gen_repetition_penalty",
        "gen_max_new_tokens",
    ],
)

display(
    summary_by_ruleset.sort_values(
        by=["model_key", "config_label", "sari", "bertscore_f1"],
        ascending=[True, True, False, False]
    ).head(30)
)

summary_by_ruleset_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_summary_by_ruleset.csv"
summary_by_ruleset.to_csv(summary_by_ruleset_path, index=False, encoding="utf-8-sig")

print(summary_by_ruleset_path)

,model_key,config_label,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,sari,bleu,fernandez_huerta_pred,...,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
4,llama3,llama3_cfg_1,R4,0.3,0.9,1.15,256.0,42.008218,13.875197,85.335956,...,0.0,0.425472,0.550758,0.426158,0.250538,0.363877,69.677437,51.983615,17.693823,0.793462
3,llama3,llama3_cfg_1,R3,0.3,0.9,1.15,256.0,41.698259,12.348385,83.094208,...,0.0,0.412065,0.533588,0.424543,0.243142,0.360349,70.138754,51.983615,18.155140,0.790053
0,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,256.0,41.540371,13.598329,82.675341,...,0.0,0.378020,0.494778,0.457421,0.269213,0.377289,65.496580,51.983615,13.512966,0.801239
1,llama3,llama3_cfg_1,R1,0.3,0.9,1.15,256.0,40.908837,13.424919,81.448009,...,0.0,0.411459,0.537371,0.454763,0.260010,0.372331,67.139742,51.983615,15.156127,0.794884
2,llama3,llama3_cfg_1,R2,0.3,0.9,1.15,256.0,40.491464,12.228606,81.345291,...,0.0,0.395276,0.545014,0.437192,0.235431,0.359579,72.247562,51.983615,20.263947,0.784669
7,llama3,llama3_cfg_2,R2,0.7,0.9,1.10,512.0,42.969879,15.394757,84.192944,...,0.0,0.352626,0.502511,0.479850,0.283303,0.398782,66.226652,51.983615,14.243037,0.805323
6,llama3,llama3_cfg_2,R1,0.7,0.9,1.10,512.0,42.769451,14.955279,83.765192,...,0.0,0.338931,0.500175,0.467946,0.286316,0.418976,69.121036,51.983615,17.137421,0.800380
5,llama3,llama3_cfg_2,R0,0.7,0.9,1.10,512.0,40.148707,12.855757,83.314215,...,0.0,0.359663,0.473849,0.446163,0.258097,0.387624,64.294319,51.983615,12.310704,0.799588
9,llama3,llama3_cfg_2,R4,0.7,0.9,1.10,512.0,39.094593,12.980107,82.038269,...,0.0,0.357469,0.539141,0.414418,0.233048,0.357504,73.724059,51.983615,21.740445,0.783524
8,llama3,llama3_cfg_2,R3,0.7,0.9,1.10,512.0,38.789174,11.250912,78.944826,...,0.0,0.396602,0.559180,0.422549,0.229502,0.352662,69.775872,51.983615,17.792258,0.783743


/home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260404_031154_summary_by_ruleset.csv


In [22]:
best_ruleset_per_config = (
    summary_by_ruleset
    .sort_values(
        by=["model_key", "config_label", "sari", "bertscore_f1"],
        ascending=[True, True, False, False]
    )
    .groupby(["model_key", "config_label"], as_index=False, group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

display(best_ruleset_per_config)

best_ruleset_per_config_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_best_ruleset_per_config.csv"
best_ruleset_per_config.to_csv(best_ruleset_per_config_path, index=False, encoding="utf-8-sig")

print(best_ruleset_per_config_path)

,model_key,config_label,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,sari,bleu,fernandez_huerta_pred,...,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
0,llama3,llama3_cfg_1,R4,0.3,0.9,1.15,256.0,42.008218,13.875197,85.335956,...,0.0,0.425472,0.550758,0.426158,0.250538,0.363877,69.677437,51.983615,17.693823,0.793462
1,llama3,llama3_cfg_2,R2,0.7,0.9,1.10,512.0,42.969879,15.394757,84.192944,...,0.0,0.352626,0.502511,0.479850,0.283303,0.398782,66.226652,51.983615,14.243037,0.805323
2,mistral,mistral_cfg_1,R2,0.7,0.9,1.10,512.0,39.754386,12.575512,83.294792,...,0.0,0.448527,0.332239,0.437225,0.239580,0.386844,51.979023,51.983615,-0.004592,0.785674
3,mistral,mistral_cfg_2,R3,0.3,0.9,1.15,400.0,39.599908,10.037956,80.720754,...,0.0,0.520015,0.359740,0.420697,0.222833,0.351828,53.887471,51.983615,1.903857,0.780018


/home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260404_031154_best_ruleset_per_config.csv


In [23]:
best_ruleset_per_model = summarize_metrics(
    evaluated_df,
    group_cols=["model_key", "ruleset"]
)

display(
    best_ruleset_per_model.sort_values(
        by=["model_key", "sari", "bertscore_f1"],
        ascending=[True, False, False]
    )
)

best_ruleset_per_model_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_best_ruleset_per_model.csv"
best_ruleset_per_model.to_csv(best_ruleset_per_model_path, index=False, encoding="utf-8-sig")

print(best_ruleset_per_model_path)

,model_key,ruleset,sari,bleu,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,sentence_splits,levenshtein_similarity,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
1,llama3,R1,41.839144,14.190099,82.606600,88.314342,-5.707742,0.781029,0.650,0.488120,0.0,0.375195,0.518773,0.461354,0.273163,0.395653,68.130389,51.983615,16.146774,0.797632
2,llama3,R2,41.730672,13.811682,82.769117,88.314342,-5.545225,0.794579,0.550,0.472634,0.0,0.373951,0.523762,0.458521,0.259367,0.379180,69.237107,51.983615,17.253492,0.794996
0,llama3,R0,40.844539,13.227043,82.994778,88.314342,-5.319564,0.839746,0.575,0.508705,0.0,0.368841,0.484313,0.451792,0.263655,0.382456,64.895450,51.983615,12.911835,0.800413
4,llama3,R4,40.551405,13.427652,83.687112,88.314342,-4.627230,0.759031,0.600,0.455108,0.0,0.391471,0.544949,0.420288,0.241793,0.360691,71.700748,51.983615,19.717134,0.788493
3,llama3,R3,40.243716,11.799649,81.019517,88.314342,-7.294825,0.781735,0.750,0.451867,0.0,0.404333,0.546384,0.423546,0.236322,0.356505,69.957313,51.983615,17.973699,0.786898
7,mistral,R2,39.500098,10.857904,81.784288,88.314342,-6.530054,1.584306,0.900,0.399441,0.0,0.489269,0.358446,0.423395,0.221942,0.365970,51.931763,51.983615,-0.051852,0.782278
5,mistral,R0,39.489801,11.310747,81.223302,88.314342,-7.091040,1.440180,0.975,0.422998,0.0,0.504177,0.361767,0.432704,0.248594,0.374922,55.874481,51.983615,3.890866,0.780860
8,mistral,R3,39.220596,9.864114,80.475484,88.314342,-7.838858,1.521020,1.125,0.382581,0.0,0.520367,0.361569,0.422085,0.221229,0.352427,56.915951,51.983615,4.932336,0.775227
9,mistral,R4,38.756330,10.936902,80.071643,88.314342,-8.242699,1.547578,0.950,0.406925,0.0,0.503344,0.354229,0.427575,0.225869,0.373244,52.937641,51.983615,0.954026,0.781019
6,mistral,R1,38.581110,9.816048,83.087255,88.314342,-5.227087,1.883192,1.250,0.350796,0.0,0.550498,0.361525,0.393450,0.212017,0.335818,56.197439,51.983615,4.213824,0.769301


/home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260404_031154_best_ruleset_per_model.csv


In [24]:
selected_configs = best_ruleset_per_config[
    ["model_key", "config_label", "ruleset"]
].drop_duplicates()

finalist_cases = evaluated_df.merge(
    selected_configs,
    on=["model_key", "config_label", "ruleset"],
    how="inner"
)

qual_cols = [
    "sample_id",
    "idFinal",
    "grupo",
    "motivo",
    "model_key",
    "config_label",
    "ruleset",
    "gen_temperature",
    "gen_top_p",
    "gen_repetition_penalty",
    "source_text",
    "reference_text",
    "generated_text",
    "sari",
    "bertscore_f1",
    "rougeL_f",
    "compression_ratio_eval",
    "exact_copy",
]

qual_cols = [c for c in qual_cols if c in finalist_cases.columns]

qual_review_df = finalist_cases[qual_cols].copy()

display(qual_review_df.head(20))

qual_review_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_qualitative_review.csv"
qual_review_df.to_csv(qual_review_path, index=False, encoding="utf-8-sig")

print(qual_review_path)

,sample_id,idFinal,grupo,motivo,model_key,config_label,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,source_text,reference_text,generated_text,sari,bertscore_f1,rougeL_f,compression_ratio_eval,exact_copy
0,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",llama3,llama3_cfg_1,R4,0.3,0.9,1.15,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,La comunidad humana más antigua se llama horda...,67.760943,0.912748,0.736842,0.900000,0
1,2976,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,llama3,llama3_cfg_1,R4,0.3,0.9,1.15,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,Prueba a guardar cada día una moneda de un dól...,32.977641,0.765994,0.166667,1.187500,0
2,3430,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,llama3,llama3_cfg_1,R4,0.3,0.9,1.15,Uno de los problemas más importantes en los ne...,Asignar precios a los productos es un problema...,La asignación de precios a los productos es un...,45.349149,0.870273,0.363636,1.058824,0
3,3679,829_LibroUAC_Sincopyright.pdf,A_cortos,Corto con número e IVA.,llama3,llama3_cfg_1,R4,0.3,0.9,1.15,"El resultado será 190, o sea, el IVA del kilo ...","El resultado será ciento noventa. Es decir, el...",El precio total del kilo de pan incluyendo el ...,42.238715,0.730420,0.303030,0.866667,0
4,3145,1045_LibroUide_Sincopyright.pdf,A_cortos,Corto con proporción financiera.,llama3,llama3_cfg_1,R4,0.3,0.9,1.15,"Por cada $100 facturados, la compañía gasta $2...",La empresa gasta dos dólares con veinte centav...,"Por cada 100 dólares vendidos, la empresa paga...",44.165738,0.797337,0.258065,1.181818,0
5,507,562_LibroUAC_Sincopyright.pdf,B_medianos,Porcentajes y rentabilidad.,llama3,llama3_cfg_1,R4,0.3,0.9,1.15,"En general, se dice que la rentabilidad normal...","Generalmente, la rentabilidad normal de una in...",La rentabilidad normal de una inversión en paí...,63.631910,0.899354,0.638298,0.750000,0
6,1756,5100_LibroBAC.pdf,B_medianos,Redacción abstracta/institucional.,llama3,llama3_cfg_1,R4,0.3,0.9,1.15,En virtud de estas y otras reflexiones fue que...,Debido a estas y otras reflexiones se diseñó e...,Este dinero es libre para pagar.,28.576764,0.680847,0.068966,0.240000,0
7,3093,875_LibroUide_Sincopyright.pdf,B_medianos,Definición contable.,llama3,llama3_cfg_1,R4,0.3,0.9,1.15,Las cuentas de activo reflejan aquello que pos...,Las cuentas de activo evidencian las pertenenc...,Las cuentas de activo muestran lo que una empr...,61.634697,0.839735,0.437500,0.937500,0
8,3192,1202_LibroUide_Sincopyright.pdf,B_medianos,"Relación de pasivo/activo, útil para precisión.",llama3,llama3_cfg_1,R4,0.3,0.9,1.15,"Por cada dólar de pasivo corriente, la compañí...",La empresa tiene noventa y tres centavos de dó...,"Por cada dólar de deuda corriente, la empresa ...",30.261851,0.768821,0.250000,0.789474,0
9,3525,230_LibroUAC_Sincopyright.pdf,B_medianos,Dato económico con cifra.,llama3,llama3_cfg_1,R4,0.3,0.9,1.15,Posee la renta per cápita más alta de Latinoam...,Tiene la renta per cápita más alta de Latinoam...,Chile tiene la renta per cápita más alta en Am...,47.278173,0.801410,0.545455,0.952381,0


/home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260404_031154_qualitative_review.csv


In [25]:
bitacora = {
    "experiment_id": EXPERIMENT_ID,
    "dataset_name": "muestra_20_ruleset_comparison",
    "n_samples": int(len(df_refine)),
    "prompt_type_fixed": PROMPT_TYPE,
    "finalist_configs": FINALIST_CONFIGS,
    "rulesets_tested": ACTIVE_RULESETS,
    "expected_total_runs": int(len(FINALIST_CONFIGS) * len(ACTIVE_RULESETS) * len(df_refine)),
    "successful_runs": int(len(successful_df)),
    "leader_metrics": ["sari", "bertscore_f1"],
    "support_metrics": ["rougeL_f", "compression_ratio_eval", "exact_copy"],
    "best_ruleset_per_config": best_ruleset_per_config.to_dict(orient="records"),
}

bitacora_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_bitacora_experimento.json"

with open(bitacora_path, "w", encoding="utf-8") as f:
    json.dump(bitacora, f, ensure_ascii=False, indent=2)

print(bitacora_path)

/home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260404_031154_bitacora_experimento.json


In [26]:
print(f"""
Resumen rapido de esta fase:

- Se probaron {len(FINALIST_CONFIGS)} configuraciones finalistas.
- Se fijo la tecnica en: {PROMPT_TYPE}
- Se compararon {len(ACTIVE_RULESETS)} rulesets: {ACTIVE_RULESETS}
- En total se esperaban {len(FINALIST_CONFIGS) * len(ACTIVE_RULESETS) * len(df_refine)} corridas.

El objetivo de esta fase fue ver como cambia el desempeño cuando se modifica el conjunto de reglas,
manteniendo fijas las configuraciones que ya habian sido seleccionadas en la etapa de hiperparametros.
La comparacion se hizo principalmente con SARI y BERTScore F1, apoyandose tambien en otras metricas
para revisar el balance entre simplificacion, preservacion del significado y nivel de compresion.
""")


Resumen rapido de esta fase:

- Se probaron 4 configuraciones finalistas.
- Se fijo la tecnica en: few-shot
- Se compararon 5 rulesets: ['R0', 'R1', 'R2', 'R3', 'R4']
- En total se esperaban 400 corridas.

El objetivo de esta fase fue ver como cambia el desempeño cuando se modifica el conjunto de reglas,
manteniendo fijas las configuraciones que ya habian sido seleccionadas en la etapa de hiperparametros.
La comparacion se hizo principalmente con SARI y BERTScore F1, apoyandose tambien en otras metricas
para revisar el balance entre simplificacion, preservacion del significado y nivel de compresion.

